# 🏭 Project 6.4 — Industrial Component Registry
**DSA for Mechatronics · Week 06 — Trees & BSTs**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A factory organises its components in a BST indexed by **component weight (grams)**.
The maintenance system needs to:
1. Find the **Lowest Common Ancestor** of two components (nearest common assembly)
2. Trace the **path from root to a target** component (assembly chain)
3. Compute the **right side view** (what a camera at the right sees — last node per level)
4. Perform a **zigzag level-order** scan (quality control camera alternates direction each pass)


## Step 1 — Build the component registry BST

In [ ]:
from collections import deque

# ── Personalised parameters ────────────────────────────────────────
N_COMPONENTS   = random.randint(14, 22)
WEIGHT_RANGE   = (random.randint(50, 200), random.randint(500, 2000))
COMP_TYPES     = ["BEARING","GEAR","SHAFT","HOUSING","SEAL","BOLT",
                  "BRACKET","PULLEY","COUPLING","SPRING","BUSHING","PIN"]

class CNode:
    def __init__(self, weight, comp_id, comp_type):
        self.weight    = weight
        self.comp_id   = comp_id
        self.comp_type = comp_type
        self.left      = None
        self.right     = None
    def __repr__(self): return f"C{self.comp_id}({self.comp_type},{self.weight}g)"

class ComponentBST:
    def __init__(self):
        self.root = None

    def insert(self, weight, comp_id, comp_type):
        """Standard BST insert by weight."""
        def _ins(node):
            if not node: return CNode(weight, comp_id, comp_type)
            if weight < node.weight:  node.left  = _ins(node.left)
            elif weight > node.weight: node.right = _ins(node.right)
            return node
        self.root = _ins(self.root)

    def inorder(self):
        out, def_stack = [], []
        n = self.root
        while n or def_stack:
            while n: def_stack.append(n); n = n.left
            n = def_stack.pop()
            out.append(n)
            n = n.right
        return out

    def height(self):
        def _h(n): return 0 if not n else 1 + max(_h(n.left), _h(n.right))
        return _h(self.root)

# Generate components
weights   = sorted(random.sample(range(*WEIGHT_RANGE), N_COMPONENTS))
ins_order = weights[:]
random.shuffle(ins_order)
comp_types = [random.choice(COMP_TYPES) for _ in range(N_COMPONENTS)]
type_map   = {w: t for w, t in zip(weights, comp_types)}

reg = ComponentBST()
for i, w in enumerate(ins_order):
    reg.insert(w, i+1, type_map[w])

all_nodes = reg.inorder()
print(f"Component registry parameters:")
print(f"  Components     : {N_COMPONENTS}")
print(f"  Weight range   : {WEIGHT_RANGE[0]} – {WEIGHT_RANGE[1]} g")
print(f"  BST height     : {reg.height()}")
print(f"  Insertion order: {ins_order[:6]}... g")
print()
print("Inorder (sorted by weight):")
print(f"  {'#':>3}  {'Weight':>8}  {'Type':<10}  ID")
print(f"  {'─'*3}  {'─'*8}  {'─'*10}  {'─'*4}")
for i, n in enumerate(all_nodes):
    print(f"  {i+1:3d}  {n.weight:8d}g  {n.comp_type:<10}  C{n.comp_id}")


## Step 2 — Lowest Common Ancestor (LCA)

In [ ]:
def lca_bst(root, w1, w2):
    """
    In a BST, LCA is easy to find without storing parent pointers:
    - If both w1 and w2 are smaller  → LCA is in left subtree
    - If both w1 and w2 are larger   → LCA is in right subtree
    - Otherwise                      → current node IS the LCA
    This is O(h) time, O(1) space (iterative).
    """
    node = root
    while node:
        if w1 < node.weight and w2 < node.weight:
            node = node.left
        elif w1 > node.weight and w2 > node.weight:
            node = node.right
        else:
            return node   # divergence point = LCA
    return None

def path_to_node(root, target_weight):
    """Trace the path from root to the target node in a BST. O(h)."""
    path = []
    node = root
    while node:
        path.append(node)
        if target_weight == node.weight: return path
        node = node.left if target_weight < node.weight else node.right
    return []   # not found

# Pick two components for LCA query
w1 = all_nodes[random.randint(0, N_COMPONENTS // 3)].weight
w2 = all_nodes[random.randint(2 * N_COMPONENTS // 3, N_COMPONENTS - 1)].weight

lca_node = lca_bst(reg.root, w1, w2)
path_w1  = path_to_node(reg.root, w1)
path_w2  = path_to_node(reg.root, w2)

print(f"Lowest Common Ancestor (LCA):")
print(f"  Component 1    : {w1}g  ({type_map[w1]})")
print(f"  Component 2    : {w2}g  ({type_map[w2]})")
print(f"  LCA            : {lca_node.weight}g  ({lca_node.comp_type})")
print()
print(f"Path from root to {w1}g:")
print(f"  {' → '.join(f'{n.weight}g' for n in path_w1)}")
print(f"Path from root to {w2}g:")
print(f"  {' → '.join(f'{n.weight}g' for n in path_w2)}")
print(f"\n  Path lengths: {len(path_w1)} and {len(path_w2)} hops from root")


## Step 3 — Right side view

In [ ]:
def right_side_view(root):
    """
    Right side view: the last node visible at each level when viewed from the right.
    Uses BFS (level-order) and takes the last node of each level.
    O(n) time and space.
    """
    if not root: return []
    view = []
    q    = deque([root])
    while q:
        width = len(q)
        for i in range(width):
            node = q.popleft()
            if i == width - 1:         # last node in this level
                view.append(node)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
    return view

rsv = right_side_view(reg.root)

print(f"Right side view (last node visible at each level):")
print(f"  {'Level':>6}  {'Weight':>8}  {'Type':<10}  ID")
print(f"  {'─'*6}  {'─'*8}  {'─'*10}  {'─'*4}")
for lv, node in enumerate(rsv):
    print(f"  {lv:6d}  {node.weight:8d}g  {node.comp_type:<10}  C{node.comp_id}")

print(f"\n  Camera sees {len(rsv)} components total  ({len(rsv)} levels)")


## Step 4 — Zigzag level-order scan

In [ ]:
def zigzag_level_order(root):
    """
    Zigzag (snake) level-order traversal:
    Odd levels → left-to-right
    Even levels → right-to-left
    Uses a deque: append to right normally; appendleft when reversing.
    Returns list of lists, one per level.
    """
    if not root: return []
    result  = []
    q       = deque([root])
    left_to_right = True
    while q:
        width = len(q)
        level = deque()
        for _ in range(width):
            node = q.popleft()
            if left_to_right:
                level.append(node.weight)
            else:
                level.appendleft(node.weight)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
        result.append(list(level))
        left_to_right = not left_to_right
    return result

zigzag = zigzag_level_order(reg.root)

print(f"Zigzag level-order scan ({len(zigzag)} levels):")
print(f"  {'Level':>6}  {'Direction':<12}  Weights (g)")
print(f"  {'─'*6}  {'─'*12}  {'─'*40}")
for lv, row in enumerate(zigzag):
    direction = "→ L-to-R" if lv % 2 == 0 else "← R-to-L"
    print(f"  {lv:6d}  {direction:<12}  {row}")

# Summary statistics
all_weights = [n.weight for n in all_nodes]
total_weight = sum(all_weights)
avg_weight   = round(total_weight / N_COMPONENTS, 1)
heaviest     = all_nodes[-1]
lightest     = all_nodes[0]

print(f"\nRegistry summary:")
print(f"  Total weight   : {total_weight} g")
print(f"  Average weight : {avg_weight} g")
print(f"  Lightest       : {lightest.weight}g  ({lightest.comp_type})")
print(f"  Heaviest       : {heaviest.weight}g  ({heaviest.comp_type})")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " COMPONENT REGISTRY BST — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  REGISTRY PARAMETERS" + " "*(W-21) + "║")
print(f"║  {'Components':<28}: {N_COMPONENTS:<26}║")
print(f"║  {'Weight range':<28}: {WEIGHT_RANGE[0]} – {WEIGHT_RANGE[1]} g{'':<16}║")
print(f"║  {'BST height':<28}: {reg.height():<26}║")
print(f"║  {'Total weight':<28}: {total_weight} g{'':<22}║")
print("╠" + "═"*W + "╣")
print("║  LCA QUERY" + " "*(W-11) + "║")
print(f"║  {'Component A':<28}: {w1}g  ({type_map[w1]}){'':<12}║")
print(f"║  {'Component B':<28}: {w2}g  ({type_map[w2]}){'':<12}║")
print(f"║  {'LCA (nearest assembly)':<28}: {lca_node.weight}g  ({lca_node.comp_type}){'':<10}║")
print(f"║  {'Path depths':<28}: {len(path_w1)} and {len(path_w2)} hops{'':<14}║")
print("╠" + "═"*W + "╣")
print("║  TRAVERSALS" + " "*(W-12) + "║")
print(f"║  {'Right-side view nodes':<28}: {len(rsv):<26}║")
print(f"║  {'Rightmost component':<28}: {rsv[-1].weight}g  ({rsv[-1].comp_type}){'':<10}║")
print(f"║  {'Zigzag levels':<28}: {len(zigzag):<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the system?

*Your answer here:*

---

**Q2 — Which tree concept did you find most important, and why?**
Pick one technique from the notebook (traversal, BST property, recursion pattern…) and explain in your own words what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
